# Where should a limited casework team spend next week's time?

**An independent decision study by Abdulaziz Aldoseri.**

A hypothetical telecom complaint-casework team has limited staff with different skills. Before each week begins, it must allocate whole cross-trained employees across Billing and commercial, Service and technical, and Privacy workstreams. Dedicated specialists stay in their eligible workstream. Unprocessed cases carry into the next week.

The observed records are consumer submissions to the US Federal Communications Commission, **not an operator's contact-centre queue**. The source has no observed employees, handling times, completion times or actual backlog. Staff, skills and processing rates are explicit scenarios. A model case-processing slot is not proof of complaint resolution, customer satisfaction or an achieved response deadline.

The question is whether balanced queue coverage is worth a different allocation of scarce processing capacity. We compare a fixed staffing split, an adaptive workload split, a current-week exact balanced allocation and a current-week actual-intake reference. All policies have the same feasible staff pools and their own simulated backlogs.

**Source:** [Federal Communications Commission, CGB — Consumer Complaints Data](https://opendata.fcc.gov/Consumer/CGB-Consumer-Complaints-Data/3xyp-aqkj), public domain US Government work. Modifications: fixed Phone/Internet cohort, documented issue mapping and exclusions, UTC daily/weekly aggregation, and independent hypothetical staffing simulation. Consumer-selected allegations are not independently verified by FCC. No endorsement is implied.


## Open the complete study

Run the cells in order in a fresh CPU runtime. Setup automatically retrieves the complete, frozen study package and verifies its exact size and SHA-256 before extracting or importing project code. Data, model code, source attribution, tests and full evaluated results are included; no separate file download, upload or Drive access is needed. Prepared or aggregate data retains the scope documented below and in the source register; it is not a claim that every publisher's raw export is redistributed.

Google Colab setup installs this study's pinned requirements where required. For local Jupyter, install the package requirements first. The acquisition and extraction path has been tested locally, including corruption, retrieval failure and rerun checks. Hosted Colab execution has not been verified. An unpublished website preview's Colab link becomes available when the notebook is published to GitHub.

This delivery edition changes setup only. The original scientific notebook, code, data, evaluation protocol and results remain in the unchanged reproduction package. The setup cell exposes the exact package URL and checksum for inspection.


In [ ]:
"""Standard-library ZIP validation for the telecom staffing reproduction notebook.

Checksums detect corruption and manifest inconsistency, not publisher identity.
The delivery setup pins the complete archive by size and SHA-256 before this
helper runs. No archive code is executed by this helper.
"""
from __future__ import annotations

from hashlib import sha256
from io import BytesIO
import json
from pathlib import Path, PurePosixPath
import re
import stat
from zipfile import ZipFile

MAX_ARCHIVE_BYTES = 100 * 1024 * 1024
MAX_TOTAL_BYTES = 200 * 1024 * 1024
MAX_FILE_BYTES = 32 * 1024 * 1024
MAX_FILES = 2000
MANIFEST_NAME = "package_manifest.json"


def safe_name(name: str) -> str:
    if not isinstance(name, str) or not name or "\\" in name or ":" in name or "\x00" in name:
        raise ValueError("Invalid archive path")
    path = PurePosixPath(name)
    if path.is_absolute() or any(part in {"", ".", ".."} for part in name.split("/")):
        raise ValueError("Archive path must be relative without traversal")
    if any(part.casefold() in {"private", ".git", ".env", "source", "attempts"} for part in path.parts):
        raise ValueError("Private or repository-internal material is not allowed in the public package")
    if path.suffix.casefold() in {".xlsx", ".xls"}:
        raise ValueError("The public package must not contain raw source workbooks")
    return path.as_posix()


def manifest_entries(raw: bytes) -> dict:
    if len(raw) > 1024 * 1024:
        raise ValueError("Manifest exceeds size limit")
    value = json.loads(raw.decode("utf-8"))
    if not isinstance(value, dict) or not isinstance(value.get("files"), list):
        raise ValueError("Manifest must contain a files array")
    if not 1 <= len(value["files"]) <= MAX_FILES:
        raise ValueError("Invalid manifest file count")
    records, folded = {}, set()
    for item in value["files"]:
        if not isinstance(item, dict):
            raise ValueError("Invalid manifest record")
        name = safe_name(item.get("path"))
        size, digest = item.get("bytes"), item.get("sha256")
        if name == MANIFEST_NAME or name.casefold() in folded:
            raise ValueError("Duplicate, case-colliding or self-referencing manifest path")
        if isinstance(size, bool) or not isinstance(size, int) or not 0 <= size <= MAX_FILE_BYTES:
            raise ValueError("Invalid manifest size")
        if not isinstance(digest, str) or not re.fullmatch(r"[0-9a-f]{64}", digest):
            raise ValueError("Invalid SHA-256 digest")
        records[name] = {"bytes": size, "sha256": digest}
        folded.add(name.casefold())
    if sum(record["bytes"] for record in records.values()) > MAX_TOTAL_BYTES:
        raise ValueError("Manifest total exceeds size limit")
    return records


def verify_package(directory: str | Path, allow_notebook_edits: bool = False) -> dict:
    """Verify declared files; optional working-notebook edits do not exempt code/data.

    Extraction always uses strict verification. At notebook runtime, Jupyter
    saves outputs and edited parameters into .ipynb files, so those working
    documents may differ while all executable modules/data/requirements remain
    hash-checked. Notebook paths still must exist and be regular bounded files.
    """
    root = Path(directory).resolve()
    manifest = root / MANIFEST_NAME
    if manifest.is_symlink() or not manifest.is_file():
        raise ValueError("Package manifest is missing or is a symlink")
    entries = manifest_entries(manifest.read_bytes())
    for name, item in entries.items():
        target = root / name
        if any(part.is_symlink() for part in [target, *target.parents] if part != root.parent):
            raise ValueError("Symlinks are not permitted in a package")
        if not target.resolve().is_relative_to(root) or not target.is_file():
            raise ValueError("Missing or unsafe package file")
        if allow_notebook_edits and target.suffix.casefold() == ".ipynb":
            if target.stat().st_size > MAX_FILE_BYTES:
                raise ValueError("Working notebook exceeds size limit")
            continue
        if target.stat().st_size != item["bytes"] or sha256(target.read_bytes()).hexdigest() != item["sha256"]:
            raise ValueError("Package checksum or size mismatch: " + name)
    return entries


def safe_extract_package(archive: bytes | str | Path, destination: str | Path) -> Path:
    """Verify all members before writing to a new/empty destination directory."""
    if isinstance(archive, bytes):
        raw = archive
    else:
        path = Path(archive)
        if path.stat().st_size > MAX_ARCHIVE_BYTES:
            raise ValueError("Archive exceeds compressed size limit")
        raw = path.read_bytes()
    if len(raw) > MAX_ARCHIVE_BYTES:
        raise ValueError("Archive exceeds compressed size limit")
    output = Path(destination)
    if output.is_symlink():
        raise ValueError("Destination must not be a symlink")
    output = output.resolve()
    if output.exists() and (not output.is_dir() or any(output.iterdir())):
        raise ValueError("Choose a new or empty extraction directory")
    with ZipFile(BytesIO(raw)) as archive_zip:
        infos = archive_zip.infolist()
        if len(infos) > MAX_FILES + 1:
            raise ValueError("Archive has too many entries")
        names, folded, total = {}, set(), 0
        for info in infos:
            name = safe_name(info.filename.rstrip("/") if info.is_dir() else info.filename)
            if name.casefold() in folded:
                raise ValueError("Duplicate or case-colliding archive member")
            folded.add(name.casefold())
            kind = stat.S_IFMT(info.external_attr >> 16)
            if kind not in {0, stat.S_IFREG, stat.S_IFDIR} or (kind == stat.S_IFDIR and not info.is_dir()):
                raise ValueError("Archive contains a nonregular entry")
            if info.flag_bits & 1:
                raise ValueError("Encrypted archive members are not supported")
            if info.is_dir():
                continue
            if not 0 <= info.file_size <= MAX_FILE_BYTES:
                raise ValueError("Archive member exceeds size limit")
            total += info.file_size
            names[name] = info
        if total > MAX_TOTAL_BYTES:
            raise ValueError("Archive exceeds expanded size limit")
        file_names = {name.casefold() for name in names}
        prefixes = {}
        for name in names:
            parts = PurePosixPath(name).parts
            for end in range(1, len(parts) + 1):
                prefix = "/".join(parts[:end])
                folded_prefix = prefix.casefold()
                if folded_prefix in prefixes and prefixes[folded_prefix] != prefix:
                    raise ValueError("Inconsistent case in archive path components")
                prefixes[folded_prefix] = prefix
                if end < len(parts) and folded_prefix in file_names:
                    raise ValueError("Archive file conflicts with a required directory")
        if MANIFEST_NAME not in names:
            raise ValueError("Archive root must contain package_manifest.json")
        if names[MANIFEST_NAME].file_size > 1024 * 1024:
            raise ValueError("Manifest exceeds size limit")
        manifest = archive_zip.read(names[MANIFEST_NAME])
        entries = manifest_entries(manifest)
        if set(names) != set(entries) | {MANIFEST_NAME}:
            raise ValueError("Every archive file must appear exactly once in the manifest")
        verified = {MANIFEST_NAME: manifest}
        for name, item in entries.items():
            info = names[name]
            if info.file_size != item["bytes"]:
                raise ValueError("Member size disagrees with manifest")
            with archive_zip.open(info) as member:
                contents = member.read(MAX_FILE_BYTES + 1)
            if len(contents) != item["bytes"] or sha256(contents).hexdigest() != item["sha256"]:
                raise ValueError("Member checksum disagrees with manifest: " + name)
            verified[name] = contents
    # The manifest is validated before any data or executable source is written.
    output.mkdir(parents=True, exist_ok=True)
    for name, contents in verified.items():
        target = output / name
        if not target.resolve().is_relative_to(output):
            raise ValueError("Unsafe extraction target")
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open("xb") as handle:
            handle.write(contents)
    verify_package(output)
    return output

# Delivery-only setup. The scientific package remains the frozen original.
import os
import tempfile
from urllib.error import HTTPError, URLError
from urllib.request import urlopen

STUDY_ID = 'telecom'
PACKAGE_URL = 'https://raw.githubusercontent.com/Abdulaziz-Aldoseri/abdulaziz-aldoseri.github.io/master/public/files/telecom/telecom-staff-reproducibility.zip'
PACKAGE_SHA256 = 'f765b078606a389fb7b17fe629179f50ac284bec03c9880d4c944cebb36994ef'
PACKAGE_BYTES = 512303


def load_study_package(package_url=PACKAGE_URL, destination=None):
    """Fetch the complete pinned package; verify before extraction or imports.

    The optional URL/directory arguments permit local delivery verification.
    Analysis inputs and expected archive bytes are never inferred from a URL.
    """
    from hashlib import sha256
    try:
        with urlopen(package_url, timeout=30) as response:
            payload = response.read(PACKAGE_BYTES + 1)
    except (HTTPError, URLError, TimeoutError, OSError) as error:
        raise RuntimeError(
            "The study package could not be retrieved. Check the connection and "
            "rerun this cell. An unpublished preview becomes available after its "
            "GitHub release. No study code has been imported."
        ) from error
    if len(payload) != PACKAGE_BYTES or sha256(payload).hexdigest() != PACKAGE_SHA256:
        raise ValueError("The study package failed its pinned size/SHA-256 check. Nothing was extracted.")
    target = Path(destination) if destination is not None else (
        Path(tempfile.gettempdir()) / ("decision-lab-" + STUDY_ID + "-" + PACKAGE_SHA256[:16])
    )
    if target.exists():
        # Bind cached verification to the freshly pinned archive, never to a
        # mutable local manifest. Reject extra modules that could shadow imports.
        from io import BytesIO
        from zipfile import ZipFile
        if target.is_symlink() or not target.is_dir():
            raise ValueError("The cached study directory is unsafe.")
        prefix = "energy-flexibility/" if STUDY_ID == "energy" else ""
        manifest_name = "PACKAGE_MANIFEST.json" if STUDY_ID == "energy" else "package_manifest.json"
        with ZipFile(BytesIO(payload)) as pinned:
            trusted_manifest = pinned.read(prefix + manifest_name)
            allowed_roots = {name[len(prefix):].split("/", 1)[0] for name in pinned.namelist()}
        local_manifest = target / manifest_name
        if local_manifest.is_symlink() or not local_manifest.is_file() or local_manifest.read_bytes() != trusted_manifest:
            raise ValueError("The cached manifest differs from the pinned study package.")
        allowed_generated = {"__pycache__"}
        if STUDY_ID == "airline":
            allowed_generated.add("notebook_reproduction")
        if STUDY_ID == "mobility":
            allowed_generated.update({"local-source-downloads", "reproduced-aggregates"})
        for entry in target.iterdir():
            if entry.is_symlink():
                raise ValueError("A cached study entry is a symlink.")
            working_notebook = entry.is_file() and entry.suffix.lower() == ".ipynb"
            if entry.name not in allowed_roots | allowed_generated and not working_notebook:
                raise ValueError("Unexpected cached study entry could shadow code: " + entry.name)
    if target.exists():
        verify_package(target)
        return target
    return safe_extract_package(payload, target)


In [ ]:
import importlib.util
import subprocess
import sys
import tempfile
import html
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except (ImportError, ModuleNotFoundError):
    IN_COLAB = False
PROJECT = load_study_package()
verified_files = verify_package(PROJECT)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
from staff_model import Scenario, SCALE, forecast, replay
from verify_reproduction import verify
payload = json.loads((PROJECT / "outputs/browser_payload.json").read_text(encoding="utf-8"))
try:
    from IPython.display import display, HTML
except ImportError:
    def HTML(value): return value
    def display(value): print(value)
def show_table(rows):
    if not rows: return
    headers = list(rows[0])
    table = '<table><thead><tr>' + ''.join('<th>'+html.escape(str(h))+'</th>' for h in headers) + '</tr></thead><tbody>'
    for row in rows:
        table += '<tr>' + ''.join('<td>'+html.escape(str(row[h]))+'</td>' for h in headers) + '</tr>'
    display(HTML(table+'</tbody></table>'))
print(f"Verified {len(verified_files)} files. Loaded {len(payload['scenarios'])} scenarios and {len(payload['weeks'])} held-out weeks.")


## 2. Know which inputs are observed

The preparation admits 102,757 complaints from a broad 205,892-record Phone/Internet projection. It excludes 100,600 unwanted-call records, 2,433 junk-fax records and 102 records with no issue label. All 252 calendar dates have broad-source observations. The public package carries only aggregates and source hashes.

`ticket_created` is the fixed UTC submission timestamp. The separate floating publisher date and the consumer's reported occurrence date are not used for staffing arrival time. This is a retrospective event-time replay, assuming an internal planner can observe submitted intake before the next weekly cutoff. Historical API availability and revisions are unknown.

Training: 6 January–27 April 2025, 16 weeks. Validation: 28 April–22 June, 8 weeks. Test: 23 June–14 September, 12 weeks. Membership, splits, scenarios and default week were fixed before fitting. The source's later October completeness discontinuity is outside this bounded study.


In [ ]:
import csv
prep = json.loads((PROJECT / "DATA_PREPARATION.json").read_text(encoding="utf-8"))
with (PROJECT / "data/weekly_intake.csv").open(encoding="utf-8", newline="") as handle:
    weekly = list(csv.DictReader(handle))
assert len(weekly) == 36
assert [r['split'] for r in weekly] == ['training']*16 + ['validation']*8 + ['test']*12
show_table([{'Split': key, 'Weeks': value['weeks'], 'Billing': value['intake_by_queue'][0], 'Technical': value['intake_by_queue'][1], 'Privacy': value['intake_by_queue'][2]} for key, value in prep['by_split'].items()])
assert prep['independent_aggregate_reconciled'] and prep['zero_broad_source_dates'] == []


## 3. Read the choices, objective and constraints

Controls select total staff, cross-trained share and processing-capacity multiplier. Base specialist slots are 60, 40 and 30 cases per staff-week. Cross-trained staff provide 80% of that rate. Dedicated skill proportions are 45%, 45% and 10%. Every one of these workforce values is hypothetical, not measured in the FCC data.

With F cross-trained employees, the model enumerates every integer allocation `(x_billing, x_technical, x_privacy)` with nonnegative entries summing to F. Dedicated specialists remain fixed. Queue capacity is pooled and floored once to whole-case slots.

Projected cases are the policy's own opening backlog plus the forecast. The objective first minimizes the largest projected unprocessed share among nonempty queues, then total projected remaining cases, then internal staff reallocation; queue order breaks remaining ties. Empty queues have zero share. Exact enumeration establishes one-week optimality at that common state, not full-horizon or cross-policy trajectory dominance.

At week end, process at most opening backlog plus actual intake, oldest cohorts first. Unprocessed cases carry forward separately for each policy. This weekly pooling abstraction does not establish an hourly rota, measured waits or actual resolution.


## 4. Reproduce validation before held-out evaluation

Candidates are means of the preceding 1, 4, 8 or 13 complete weeks. The shared selected forecast is fixed by validation MAE, with smaller-window ties. It uses only earlier weeks. Hindsight instead receives actual current-week intake and is labelled separately; it is not a global bound.

The next cell runs a fresh validation and complete 72-scenario evaluation in a new directory. It checks input hashes, actual freeze times, validation sidecar and no-overwrite guards. The original results remain unchanged. The final comparison requires identical analytical CSV bytes, equal payload values and equal stress outputs while preserving honest new run timestamps/provenance hashes.


In [ ]:
RUN = Path(tempfile.mkdtemp(prefix="telecom-staff-reproduction-")) / "outputs"
for stage in ('validate', 'evaluate'):
    result = subprocess.run([sys.executable, str(PROJECT / 'pipeline.py'), stage, '--output', str(RUN)], cwd=PROJECT, text=True, capture_output=True, check=True)
    print(stage + ': completed')
reproduction_check = verify(RUN)
assert reproduction_check['status'] == 'PASS'
print(reproduction_check)
show_table([{'Prior weeks': v['window'], 'Validation MAE': round(v['mae'], 2)} for v in payload['forecastValidation']])
print('Frozen window:', payload['selectedForecastWindow'])
show_table([{'Prior weeks': v['window'], 'Test MAE': round(v['mae'], 2)} for v in payload['forecastTest']])


## 5. Inspect a staff allocation and its full trajectory

Change the three scenario values and selected week below. The grid values match the interactive page. Selecting a week shows one point on a complete policy trajectory; it does not reset backlog or modify only that week. Policy backlogs differ because earlier allocations differ.

Use staff `{0,24,36,48,60,72}`, cross-trained percent `{0,25,50,100}` and capacity percent `{75,100,125}`. Week index 0 is the preselected first held-out week. Counts labelled backlog or processed are simulated inside this model.


In [ ]:
STAFF = 48
CROSS_TRAINED_PERCENT = 25
CAPACITY_PERCENT = 100
WEEK_INDEX = 0
key = f'{STAFF}-{CROSS_TRAINED_PERCENT}-{CAPACITY_PERCENT}'
selected = next((s for s in payload['scenarios'] if s['id'] == key), None)
if selected is None or not 0 <= WEEK_INDEX < len(payload['weeks']):
    raise ValueError('Choose a scenario/week from the evaluated grid.')
row = selected['policies']['optimized']['weeks'][WEEK_INDEX]
print('Week:', payload['weeks'][WEEK_INDEX], '| dedicated + flexible staff:', selected['dedicated'], '+', selected['flex'])
show_table([{'Queue': q['label'], 'Dedicated': row['dedicated'][i], 'Cross-trained': row['allocation'][i], 'Forecast intake': round(payload['forecastScaled'][WEEK_INDEX][i]/SCALE,2), 'Observed intake': row['actualIntake'][i], 'Opening backlog': row['openingBacklog'][i], 'Model slots': row['capacity'][i], 'Processed in model': row['processed'][i], 'Ending backlog': row['endBacklog'][i]} for i,q in enumerate(payload['queues'])])
show_table([{'Week': payload['weeks'][w], **{p: sum(selected['policies'][p]['weeks'][w]['endBacklog']) for p in ('fixed','adaptive','optimized','hindsight')}} for w in range(12)])
show_table([{'Policy': p, 'Ending backlog': r['summary']['endingBacklog'], 'Sum weekly backlog': r['summary']['cumulativeBacklog'], 'Mean worst share (%)': round(100*r['summary']['meanWorstShare'],2), 'Internal reallocations': r['summary']['reallocated']} for p,r in selected['policies'].items()])


## 6. Retain trade-offs, weak results and failures

The default balanced policy has a lower mean worst-queue unprocessed share, but more total backlog than both baselines in all 12 weeks. Across the full grid, cumulative backlog is higher in 41 settings, lower in 4 and tied in 27 against each baseline. Its mean worst-queue share is worse in three settings against each baseline. These are scenario counts, not probabilities.

Validation selected last week, but the eight-week candidate later has lower held-out forecast error. That adverse diagnostic is retained without changing the chosen forecast. Increasing flexibility also incurs the assumed 80% processing-rate penalty; universal cross-training is not guaranteed to help.

Stress cases below are explicitly modified model scenarios. Zero staff conserves all cases; ample staff clears them. An absence changes actual available staff, and initial backlog or a surge can persist through the horizon. The original forecast path is held fixed in the exogenous surge case.


In [ ]:
show_table([{'Stress': name, 'Balanced ending backlog': value['policies']['optimized']['endingBacklog'], 'Fixed ending backlog': value['policies']['fixed']['endingBacklog'], 'Balanced mean worst share (%)': round(value['policies']['optimized']['meanWorstShare']*100,2)} for name,value in payload['stressSummaries'].items()])
print('Whole-grid comparisons:', payload['evaluationSummary']['againstBaselines'])
for s in payload['scenarios']:
    for p, policy in s['policies'].items():
        for r in policy['weeks']:
            assert sum(r['allocation']) == s['flex']
            assert all(r['openingBacklog'][q] + r['actualIntake'][q] == r['processed'][q] + r['endBacklog'][q] for q in range(3))
print('All 72 × 12 × 4 staff/case conservation records checked.')


## 7. Verify implementation and keep the decision in scope

Tests challenge the model with an independent labelled-person assignment oracle, exact fractional objectives, zero queues, skill loss, FIFO carryover, integer pool changes, absence/return, no future forecast data and freeze tampering. Tests establish the implementation's stated behavior; they do not certify a workforce recommendation.

The useful decision is to state the priority clearly: balanced queue coverage can be worth reduced total throughput, but an allocation model cannot erase an underlying capacity shortage. Before applying this structure to an actual operator, replace the hypothetical staff skills and processing rates with appropriate operational evidence and evaluate that different decision context.

See `METHODS.md`, `RESULTS.md`, `PROTOCOL.json` and the source register for exact definitions, full adverse results and provenance. Source attribution belongs with any exported chart/table: Federal Communications Commission, CGB — Consumer Complaints Data, public domain US Government work; independent aggregation and hypothetical staffing model.


In [ ]:
tests = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(PROJECT / 'tests'), '-v'], cwd=PROJECT, text=True, capture_output=True, check=True)
print(tests.stderr or tests.stdout)
print('Completed: clean reproduction, scenario inspection and independent-oracle tests. Hosted Colab UI was not part of this local check.')
